In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
#!/usr/bin/env python3
# make_all_genres_explanations_and_figs_1111_NMF.py
#
# LOGIC KEPT IDENTICAL
# Only additions:
#  - Save avg # genre-books in Top-K per user (per figure)

from pathlib import Path
from typing import Dict, Tuple, List
import re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ========= CONFIG ============================================================

ORIG_DIR = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/NMF/result/rec/top_re/1229_NMF_ORIGINAL"
)

SEARCH_ROOT = ORIG_DIR

OUT_BASE = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/NMF/result/rec/top_re/1229_NMF_ORIGINAL/output_0102"
)
OUT_BASE.mkdir(parents=True, exist_ok=True)

K_LIST = [15, 25, 35]

# ========= Genre normalization ===============================================

CANONICAL_GENRES = [
    "Adventure", "Classics", "Drama", "Fantasy", "Historical", "Horror",
    "Mystery", "Nonfiction", "Romance", "Science Fiction", "Thriller",
    "Children's", "Adult",
]

GENRE_NORM = {
    "adventure": "adventure",
    "classics": "classics",
    "drama": "drama",
    "fantasy": "fantasy",
    "historical": "historical",
    "horror": "horror",
    "mystery": "mystery",
    "nonfiction": "nonfiction",
    "romance": "romance",
    "science fiction": "science fiction",
    "thriller": "thriller",
    "children's": "children's",
    "adult": "adult",
    "children": "children's",
    "scifi": "science fiction",
}

DISPLAY_MAP = {
    "adventure": "Adventure",
    "classics": "Classics",
    "drama": "Drama",
    "fantasy": "Fantasy",
    "historical": "Historical",
    "horror": "Horror",
    "mystery": "Mystery",
    "nonfiction": "Nonfiction",
    "romance": "Romance",
    "science fiction": "Science Fiction",
    "thriller": "Thriller",
    "children's": "Children's",
    "adult": "Adult",
}

def norm_token(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s).strip()).lower()
    return GENRE_NORM.get(s, s)

def display_canonical(s: str) -> str:
    return DISPLAY_MAP.get(norm_token(s), s)

def slugify_token(x: str) -> str:
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9]+", "_", str(x))).strip("_").lower()

# ========= IO helpers ========================================================

def load_rec_csv(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp, low_memory=False)
    if "genres_all" not in df.columns:
        df["genres_all"] = ""
    for c in ["user_id", "book_id", "rank", "est_score", "original_title"]:
        if c not in df.columns:
            df[c] = pd.NA
    return df

def has_genre(cell, target: str) -> bool:
    if pd.isna(cell):
        return False
    return target in [norm_token(t) for t in str(cell).split(",")]

def metrics_for_file(df: pd.DataFrame, genre: str):
    """
    Computes:
      avg # of genre-books in Top-K per user
    """
    target = norm_token(genre)
    is_tg = df["genres_all"].apply(lambda s: has_genre(s, target))

    per_user = (
        df.assign(is_tg=is_tg)
          .groupby("user_id")["is_tg"]
          .sum()
    )

    avg_per_user = float(per_user.mean()) if not per_user.empty else 0.0
    return avg_per_user, is_tg

def per_book_ranking(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    g = df.groupby("book_id", as_index=False)
    out = g.agg(
        freq=("book_id", "size"),
        users_n=("user_id", "nunique"),
        avg_rank=("rank", "mean"),
        avg_est_score=("est_score", "mean"),
        rank1_count=("rank", lambda s: int((s == 1).sum())),
    )

    out = out.sort_values(
        ["freq", "rank1_count", "avg_rank"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    out.insert(1, "rank", out.index + 1)
    return out

# ========= Plotting ==========================================================

def plot_genre(G_disp: str,
               avg_by_k: Dict[int, float],
               out_png: Path):
    ks = sorted(avg_by_k.keys())
    values = [avg_by_k[k] for k in ks]

    plt.figure(figsize=(8.4, 4.4), dpi=160)
    plt.bar(range(len(ks)), values, width=0.6)
    plt.xticks(range(len(ks)), [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # of genre-books in Top-K per user")
    plt.title(G_disp)
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

    # ✅ SAVE AVG VALUES PER FIGURE
    txt_path = out_png.with_suffix(".avg.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        for k in ks:
            f.write(f"Top-{k}: {avg_by_k[k]:.6f}\n")

# ========= MAIN =============================================================

def main():
    OUT_ALL = OUT_BASE / "_all_genres"
    OUT_ALL.mkdir(parents=True, exist_ok=True)

    all_rows = []
    master_lines = []

    for genre in CANONICAL_GENRES:
        disp = display_canonical(genre)
        key = slugify_token(disp)

        out_dir = OUT_BASE / f"{key}_explanation"
        out_dir.mkdir(parents=True, exist_ok=True)

        avg_by_k: Dict[int, float] = {}

        for K in K_LIST:
            fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"
            if not fp.exists():
                continue

            df = load_rec_csv(fp)
            avg_user, is_tg = metrics_for_file(df, disp)
            avg_by_k[K] = avg_user

            ranked = per_book_ranking(df.loc[is_tg])
            if not ranked.empty:
                ranked["genre"] = disp
                ranked["dataset"] = f"original{K}"
                all_rows.append(ranked)

        if avg_by_k:
            plot_genre(disp, avg_by_k, out_dir / f"{key}__original.png")

        master_lines.append(disp.lower() + ":")
        for k, v in avg_by_k.items():
            master_lines.append(f"Top-{k}: {v:.6f}")
        master_lines.append("")

    with open(OUT_ALL / "summary_master.txt", "w") as f:
        f.write("\n".join(master_lines))

    if all_rows:
        pd.concat(all_rows).to_csv(
            OUT_ALL / "per_book_ranking_all.csv",
            index=False
        )

if __name__ == "__main__":
    main()
